# Rwanda HydroServer ETL Training

This exercise creates a HydroServer workspace, the Kanzenze monitoring site, its stage datastream, and then extracts public RWB field measurements and loads them through a HydroServer ETL pipeline.

**Data source:** Rwanda Water Resources Board, Kanzenze station `259501`.

## 1. Import packages

In [ ]:
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from hydroserverpy import HydroServer
from hydroserverpy.etl import ETLPipeline
from hydroserverpy.etl.extractors import LocalFileExtractor
from hydroserverpy.etl.loaders import HydroServerLoader
from hydroserverpy.etl.transformers import ETLDataMapping, ETLTargetPath, CSVTransformer

## 2. Connect to HydroServer

You need an account on the HydroServer instance. Your password is requested securely and is not stored in the notebook.

In [ ]:
hydroserver_host = input("HydroServer URL [https://playground.hydroserver.org]: ").strip()
hydroserver_host = hydroserver_host or "https://playground.hydroserver.org"
hydroserver_email = input("HydroServer email: ").strip()
hydroserver_password = getpass("HydroServer password: ")

hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password,
)
print("Connected to HydroServer.")

## 3. Create a workspace

In [ ]:
participant_name = input("Your name or team name: ").strip()
workspace_name = f"{participant_name} - Rwanda HydroServer Training"

workspace = hs.workspaces.create(
    name=workspace_name,
    is_private=True,
)
print("Workspace:", workspace.name)
print("Workspace UUID:", workspace.uid)

## 4. Create the Kanzenze monitoring site

In [ ]:
kanzenze = hs.things.create(
    workspace=workspace,
    name="Kanzenze Hydrology Station",
    description=("Rwanda Water Resources Board station on the left bank "
                 "upstream of Kanzenze Bridge."),
    sampling_feature_type="Site",
    sampling_feature_code="RWB-259501",
    site_type="Stream",
    is_private=True,
    latitude=-2.0613,
    longitude=30.0877,
    admin_area_1="Eastern Province",
    admin_area_2="Bugesera District",
    country="RW",
    data_disclaimer="RWB observations may be provisional and subject to revision.",
)
print("Site UUID:", kanzenze.uid)
print(f"Site page: {hydroserver_host}/sites/{kanzenze.uid}")

## 5. Create stage metadata

In [ ]:
stage_property = hs.observedproperties.create(
    workspace=workspace,
    name="Stage",
    definition="http://vocabulary.odm2.org/variablename/gageHeight/",
    description="Height of the water surface above a local reference datum.",
    observed_property_type="Hydrology",
    code="STAGE",
)

metre = hs.units.create(
    workspace=workspace,
    name="Metre",
    symbol="m",
    definition="https://qudt.org/vocab/unit/M",
    unit_type="Length",
)

stage_sensor = hs.sensors.create(
    workspace=workspace,
    name="Kanzenze stage measurement method",
    description="Public RWB stage measurements displayed on the station page.",
    encoding_type="application/json",
    method_type="Instrument deployment",
    manufacturer="Rwanda Water Resources Board",
    sensor_model="Staff gauge and pressure sensor",
    method_code="RWB-259501-STAGE",
)

raw_level = hs.processinglevels.create(
    workspace=workspace,
    code="0",
    definition="Raw",
    explanation="Data have not been processed or quality controlled in this exercise.",
)
print("Stage metadata created.")

## 6. Create the stage datastream

In [ ]:
stage_datastream = hs.datastreams.create(
    name="Stage at Kanzenze Hydrology Station",
    description="Public RWB stage measurements for station 259501.",
    thing=kanzenze,
    sensor=stage_sensor,
    observed_property=stage_property,
    processing_level=raw_level,
    unit=metre,
    observation_type="Field Observation",
    result_type="Timeseries",
    sampled_medium="Surface Water",
    no_data_value=-9999.0,
    aggregation_statistic="Instantaneous",
    time_aggregation_interval=0,
    time_aggregation_interval_unit="minutes",
    status="Ongoing",
    is_private=True,
    is_visible=True,
)
print("Datastream UUID:", stage_datastream.uid)

## 7. Extract and transform public RWB observations

The station page returns HTML, so Pandas extracts its public field-visit table. This does **not** bypass or submit the form used for the larger telemetry downloads.

In [ ]:
source_url = "https://waterportal.rwb.rw/index.php/location_ng_info/259501"
tables = pd.read_html(source_url)
field_table = next(
    table for table in tables
    if {"Date", "Parameter", "Value"}.issubset(table.columns)
)
stage = field_table[
    field_table["Parameter"].astype(str).str.casefold().eq("stage")
].copy()
stage["timestamp"] = pd.to_datetime(stage["Date"], utc=True)
stage["value"] = pd.to_numeric(stage["Value"], errors="coerce")
stage = (stage[["timestamp", "value"]]
         .dropna().drop_duplicates().sort_values("timestamp"))
print(f"Prepared {len(stage)} stage observations.")
stage.head()

In [ ]:
stage.plot(x="timestamp", y="value", marker="o", figsize=(12, 5), legend=False)
plt.title("Kanzenze public stage measurements")
plt.xlabel("Time")
plt.ylabel("Stage (m)")
plt.grid(True)
plt.show()

## 8. Save a standard CSV and run the HydroServer ETL pipeline

In [ ]:
csv_path = Path("rwanda_kanzenze_stage.csv")
stage.to_csv(csv_path, index=False)

extractor = LocalFileExtractor(source_uri=str(csv_path))
transformer = CSVTransformer(
    timestamp_key="timestamp",
    delimiter=",",
    header_row=1,
    data_start_row=2,
)
loader = HydroServerLoader(client=hs, chunk_size=5000)
mappings = [
    ETLDataMapping(
        source_identifier="value",
        target_paths=[ETLTargetPath(target_identifier=str(stage_datastream.uid))],
    )
]
pipeline = ETLPipeline(extractor=extractor, transformer=transformer, loader=loader)
context = pipeline.run(data_mappings=mappings, raise_on_error=False)
print("ETL status:", context.status)
print("ETL stage:", context.stage)
if context.error:
    print("ETL error:", context.error)
if context.results:
    print("Observations loaded:", context.results.values_loaded_total)

## 9. Verify observations in HydroServer

In [ ]:
loaded = stage_datastream.get_observations(fetch_all=True)
print(f"HydroServer contains {len(loaded.dataframe)} observations.")
loaded.dataframe.tail()

## Summary

You created a workspace, monitoring site, metadata, and stage datastream; extracted and transformed public RWB field measurements; and loaded them into HydroServer through an ETL pipeline. The form-protected continuous telemetry series is outside the scope of this exercise.